# ROP Detection Training

This notebook trains a ResNet-50 model to detect Retinopathy of Prematurity (ROP) in retinal images.

## Setup

First, let's install the required dependencies and set up our environment.

In [ ]:
!pip install -q torch torchvision opencv-python-headless scikit-learn tqdm

In [ ]:
import os
import sys
import glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Dataset Preparation

Let's prepare our dataset for training.

In [ ]:
# Define dataset path (Kaggle mounts datasets to /kaggle/input/)
DATASET_PATH = "/kaggle/input/retinal-image-dataset-of-infants-and-rop"

print("Checking dataset path...")
if os.path.exists(DATASET_PATH):
    print(f"Dataset found at {DATASET_PATH}")
else:
    print(f"Dataset not found at {DATASET_PATH}")
    print("Please ensure the dataset is properly attached to this notebook.")

In [ ]:
def load_dataset(dataset_dir, sample_size=None):
    """
    Load and parse the ROP dataset from directory structure
    """
    print("Finding dataset files...")
    image_paths = glob.glob(os.path.join(dataset_dir, "**/*.jpg"), recursive=True)

    if len(image_paths) == 0:
        raise ValueError(f"No images found in {dataset_dir}")

    print(f"Found {len(image_paths)} images.")

    if sample_size and sample_size < len(image_paths):
        print(f"Sampling {sample_size} images from dataset")
        import random
        image_paths = random.sample(image_paths, sample_size)

    labels = []
    valid_paths = []

    # Parse labels from filenames. Format: ..._DG<code>_...
    for path in image_paths:
        basename = os.path.basename(path)
        # Find the DG part
        parts = basename.split("_")
        dg_part = [p for p in parts if p.startswith("DG")]
        if dg_part:
            try:
                dg_code = int(dg_part[0].replace("DG", ""))
                # Map DG codes to classes:
                # DG 11 (Physiological) and 0 to Class 0 (Normal)
                # DG 1,2,3,4,5 to Class 1 (ROP Pathological)
                if dg_code in [0, 11]:
                    labels.append(0)
                    valid_paths.append(path)
                elif dg_code in [1, 2, 3, 4, 5]:
                    labels.append(1)
                    valid_paths.append(path)
            except ValueError:
                print(f"Warning: Could not parse DG code from {basename}")

    print(f"Filtered to {len(valid_paths)} images with clear Normal/ROP labels.")
    return valid_paths, labels

# Load dataset (using a sample for faster testing)
image_paths, labels = load_dataset(DATASET_PATH, sample_size=500)  # Sample 500 images for testing

## Data Preprocessing

Define our custom dataset class with preprocessing.

In [ ]:
class ROPDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def preprocess_image(self, img):
        """
        Apply preprocessing inspired by the Nature article: ReLeSeT tool
        """
        # 1. Convert to RGB (OpenCV loads in BGR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 2. Extract Red and Green channels, zero out Blue for noise reduction
        r, g, b = cv2.split(img_rgb)
        b = np.zeros_like(b)
        img_rg = cv2.merge([r, g, b])

        # 3. Bilateral Filtering for edge-preserving smoothing
        img_filtered = cv2.bilateralFilter(img_rg, d=9, sigmaColor=3, sigmaSpace=0.1)

        # 4. CLAHE (Adaptive Histogram Equalization) on the luminance/intensity
        lab = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2LAB)
        l, a, _b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, _b))
        final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

        return final_img

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            # Fallback if image fails to load
            print(f"Warning: Could not load image {img_path}, using blank image")
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (224, 224))
            img = self.preprocess_image(img)

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

## Model Definition

Create our ResNet-50 model for ROP classification.

In [ ]:
def create_model(device, pretrained=True):
    """
    Create and configure the ResNet-50 model for ROP classification
    """
    if pretrained:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet50()

    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    model = model.to(device)
    return model

# Create model
model = create_model(device)
print("Model created successfully!")

## Training Setup

Prepare data loaders and training parameters.

In [ ]:
def create_data_loaders(X_train, X_test, y_train, y_test, batch_size, num_workers=0):
    """
    Create training and test data loaders with appropriate transforms
    """
    # Transforms
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = ROPDataset(X_train, y_train, transform=train_transform)
    test_dataset = ROPDataset(X_test, y_test, transform=test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, test_loader

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Training set: {len(X_train)} | Test set: {len(X_test)}")

# Create data loaders
train_loader, test_loader = create_data_loaders(
    X_train, X_test, y_train, y_test, batch_size=8
)

## Training Loop

Train the model with progress tracking.

In [ ]:
def train_model(model, train_loader, test_loader, device, epochs, learning_rate, save_path="best_rop_model.pth"):
    """
    Train the ROP detection model
    """
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_acc = 0.0

    print(f"Starting training for {epochs} epochs...")

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0

        # Use tqdm for progress bar
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for inputs, targets in progress_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Validation phase
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs).squeeze()
                preds = (outputs > 0.5).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(targets.numpy())

        acc = accuracy_score(all_targets, all_preds)
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f} - Acc: {acc:.4f}")

        # Save best model
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), save_path)
            print(f"New best model saved with accuracy: {acc:.4f}")

    print(f"Training completed. Best accuracy: {best_acc:.4f}")
    return model

# Train the model
trained_model = train_model(model, train_loader, test_loader, device, epochs=5, learning_rate=1e-4)

## Model Evaluation

Evaluate the trained model on the test set.

In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Evaluate the model and print detailed metrics
    """
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc="Evaluating"):
            inputs = inputs.to(device)
            outputs = model(inputs).squeeze()
            preds = (outputs > 0.5).cpu().numpy()
            probs = outputs.cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.numpy())
            all_probs.extend(probs)

    acc = accuracy_score(all_targets, all_preds)
    print(f"\nTest Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, target_names=["Normal", "ROP"], zero_division=0))

    return acc

# Evaluate the model
final_accuracy = evaluate_model(trained_model, test_loader, device)
print(f"\nFinal model accuracy: {final_accuracy:.4f}")

## Save Model

Save the trained model for future use.

In [ ]:
# Save the final model
torch.save(trained_model.state_dict(), "/kaggle/working/rop_detection_model.pth")
print("Model saved successfully to /kaggle/working/rop_detection_model.pth")

## Conclusion

We have successfully trained a ResNet-50 model to detect Retinopathy of Prematurity in retinal images. The model achieved an accuracy of {:.4f} on the test set.

The trained model has been saved and can be used for inference on new retinal images.

### Next Steps

1. Experiment with different architectures and hyperparameters
2. Increase the training dataset size for better generalization
3. Implement cross-validation for more robust evaluation
4. Try ensemble methods for improved performance
5. Deploy the model as an API for clinical use